# 07 · E2 under protocol amendment v2: scaled phase encoding

**Runtime:** T4 GPU. **Time:** about 10 minutes. **Resumable.**
Amendment: `research_question_v2.md` (committed 2026-11-13, before this run); log: `deviations.md`.

**Change.** F's encoding becomes φ = 2π·(tanh(h/τ) + 1)/2 with knob τ ∈ {1, 4, 16, 64}
(v1 was φ = β·(tanh(h) + 1)/2 with knob β). Same optics, seeds, λ grid, paired subsamples, selection and
statistics as E2. Only the F2 maps run; every other row (and v1 F, for comparison) is reused from E2.

**Why.** The E2 diagnostic showed the optics simulation matches theory exactly and that F's loss came
from tanh squashing the standardized features (h 51.7% → tanh(h) 46.1%).

**Prediction written before running.** At full data F2's accuracy rises with τ and approaches A (51.7%)
at τ = 64. Whether F2 beats A or the best digital map at N = 100 is what this run tests (gate G2 under v2).

In [ ]:
# ---- Setup (edit the two lines below once) ------------------------------------------
GITHUB_REPO = "surensagh/PhotonicsAI-Project"                    # repo that holds this code
WORKSPACE   = "/content/drive/MyDrive/optical_lm_workspace"      # splits, cache, runs, reports
# -------------------------------------------------------------------------------------
import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(WORKSPACE, exist_ok=True)
os.environ["OPTLM_WORKSPACE"] = WORKSPACE
os.environ["HF_HOME"] = "/content/hf_cache"          # downloads stay on the fast local disk

# Private repo: add a GitHub token as a Colab secret named GH_TOKEN (key icon in the left bar).
try:
    from google.colab import userdata
    token = userdata.get("GH_TOKEN")
except Exception:
    token = None
url = f"https://{token}@github.com/{GITHUB_REPO}.git" if token else f"https://github.com/{GITHUB_REPO}.git"
REPO_DIR = "/content/" + GITHUB_REPO.split("/")[-1]
cmd = ["git", "clone", "-q", url, REPO_DIR] if not os.path.isdir(REPO_DIR) else ["git", "-C", REPO_DIR, "pull", "-q", "--ff-only"]
res = subprocess.run(cmd, capture_output=True, text=True)
if res.returncode != 0:
    msg = res.stderr.replace(token, "***") if token else res.stderr
    raise RuntimeError("git failed (check GITHUB_REPO / GH_TOKEN):\n" + msg)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("repo   :", REPO_DIR)
print("commit :", subprocess.check_output(["git", "rev-parse", "HEAD"]).decode().strip())
print("dirty  :", bool(subprocess.check_output(["git", "status", "--porcelain", "--untracked-files=no"]).strip()))
print("workspace:", WORKSPACE)

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
print(torch.cuda.get_device_name(0))

In [ ]:
!python -m pytest -q tests/test_featuremaps.py

In [ ]:
!python scripts/run_e2.py --config configs/e2_v2.yaml

In [ ]:
print(open(os.path.join(WORKSPACE, "reports", "e2v2_table2.md"), encoding="utf-8").read())

In [ ]:
from IPython.display import Image, display
display(Image(os.path.join(WORKSPACE, "reports", "e2v2_fig2.png")))